# LeetCode 810 — Chalkboard XOR Game bằng Python

- Link: https://leetcode.com/problems/chalkboard-xor-game/
- Độ khó: **Hard**
- Chủ đề: Lý thuyết trò chơi (thắng/thua), Minimax + memo trên bitmask, Phép XOR, Chứng minh bằng tính chẵn lẻ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def xorGame(self, nums: List[int]) -> bool:
```

## 1. Giới thiệu & mục tiêu học

Trên bảng có các số `nums`. Alice và Bob lần lượt xoá **một** số (Alice đi trước). Nếu sau khi một người xoá mà XOR của các số còn lại bằng `0`, người đó **thua**. Nếu đầu lượt của ai đó XOR các số trên bảng đã bằng `0`, người đó **thắng**. Hỏi Alice có thắng không (hai người chơi tối ưu)?

Đáp án cuối cùng gói gọn trong **một dòng**: Alice thắng ⇔ XOR ban đầu bằng 0 **hoặc** số phần tử chẵn. Nhưng để đi tới (và tin vào) dòng đó, ta cần: mô phỏng trò chơi bằng **minimax**, quan sát quy luật từ máy, rồi **chứng minh** bằng một lập luận phản chứng gọn gàng về XOR.

Sau notebook này bạn sẽ:

1. Mô hình hoá trò chơi hai người thành hàm `win(trạng_thái)` (thế thắng/thế thua).
2. Cài minimax với **memo trên bitmask** và trên **multiset chuẩn hoá**.
3. Dùng máy để **dò quy luật** trên hàng nghìn input nhỏ.
4. Chứng minh: với số phần tử chẵn và XOR ≠ 0, luôn có nước đi an toàn.
5. Rút ra lời giải `O(n)`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên `nums` biểu diễn các số trên bảng. Alice và Bob luân phiên, Alice đi trước. Mỗi lượt, người chơi **phải** xoá đúng một số trên bảng.
- Nếu việc xoá làm XOR của **các số còn lại** bằng `0`, người vừa xoá **thua**. (XOR của bảng rỗng là `0`.)
- Nếu ở **đầu lượt** của một người, XOR các số trên bảng bằng `0`, người đó **thắng**.

Trả về `True` nếu Alice thắng khi cả hai chơi tối ưu.

**Ví dụ 1**

```text
Input: nums = [1,1,2]
Output: false
Giải thích: Alice có 2 lựa chọn:
- Xoá 1 → [1,2], XOR = 3. Bob xoá 2 → [1], XOR = 1 ≠ 0... rồi Alice phải xoá 1 → bảng rỗng XOR 0 → Alice thua.
  (Bob cũng có thể xoá 1 → [2]... nhưng Bob chọn nước tốt cho mình.)
- Xoá 2 → [1,1], XOR = 0 → Alice thua ngay.
```

**Ví dụ 2**

```text
Input: nums = [0,1]
Output: true
Giải thích: số phần tử chẵn → Alice luôn có nước an toàn.
```

**Ví dụ 3**

```text
Input: nums = [1,2,3]
Output: true
Giải thích: 1 ^ 2 ^ 3 = 0 ngay từ đầu → Alice thắng.
```

**Ràng buộc**

- `1 <= nums.length <= 1000`
- `0 <= nums[i] < 2¹⁶`

In [1]:
from typing import List
from functools import reduce
from operator import xor

EXAMPLES = [
    ([1, 1, 2], False),
    ([0, 1], True),
    ([1, 2, 3], True),
    ([5], False),
    ([0], True),
    ([2, 2, 2], False),
]
for a, exp in EXAMPLES:
    print(a, "XOR =", reduce(xor, a, 0), "| n =", len(a), "| kỳ vọng:", exp)

[1, 1, 2] XOR = 2 | n = 3 | kỳ vọng: False
[0, 1] XOR = 1 | n = 2 | kỳ vọng: True
[1, 2, 3] XOR = 0 | n = 3 | kỳ vọng: True
[5] XOR = 5 | n = 1 | kỳ vọng: False
[0] XOR = 0 | n = 1 | kỳ vọng: True
[2, 2, 2] XOR = 2 | n = 3 | kỳ vọng: False


## 3. Kiến thức nền

### 3.1. Phép XOR `^`

Tính chất: `a ^ a = 0`, `a ^ 0 = a`, giao hoán, kết hợp. Do đó XOR của cả mảng không phụ thuộc thứ tự. Nếu `X = XOR(tất cả)` thì XOR sau khi bỏ `aᵢ` là `X ^ aᵢ` (vì `X ^ aᵢ` "huỷ" `aᵢ` đi).

In [2]:
a = [5, 3, 6, 3]
X = reduce(xor, a, 0)
for i, v in enumerate(a):
    rest = reduce(xor, a[:i] + a[i + 1:], 0)
    print(f"bỏ a[{i}]={v}: XOR còn lại = {rest}, X ^ a[i] = {X ^ v}")

bỏ a[0]=5: XOR còn lại = 6, X ^ a[i] = 6
bỏ a[1]=3: XOR còn lại = 0, X ^ a[i] = 0
bỏ a[2]=6: XOR còn lại = 5, X ^ a[i] = 5
bỏ a[3]=3: XOR còn lại = 0, X ^ a[i] = 0


### 3.2. Thế thắng / thế thua (trò chơi tổ hợp)

Trong trò chơi hai người, thông tin đầy đủ, không may rủi, hữu hạn:
- Một trạng thái là **thế thắng** (cho người sắp đi) nếu **tồn tại** nước đi dẫn tới thế thua của đối thủ.
- Là **thế thua** nếu **mọi** nước đi đều dẫn tới thế thắng của đối thủ (hoặc không có nước đi hợp lệ "an toàn").

Đây là đệ quy minimax rút gọn (giá trị chỉ là thắng/thua).

In [3]:
# Ví dụ đồ chơi: bốc 1 hoặc 2 viên từ đống n viên, ai không bốc được thì thua
from functools import lru_cache

@lru_cache(None)
def toy_win(n):
    return any(not toy_win(n - t) for t in (1, 2) if t <= n)

print([toy_win(n) for n in range(10)])   # thua khi n % 3 == 0

[False, True, True, False, True, True, False, True, True, False]


### 3.3. Bitmask biểu diễn tập phần tử còn lại

Với `n ≤ 15`, trạng thái "các phần tử còn trên bảng" là một `mask` `n` bit. Xoá phần tử `i`: `mask & ~(1 << i)`. Số trạng thái `2ⁿ`.

In [4]:
mask = 0b1011
print([i for i in range(4) if mask >> i & 1], "->", bin(mask & ~(1 << 1)))

[0, 1, 3] -> 0b1001


### 3.4. `lru_cache` cho memo

`@lru_cache(None)` nhớ kết quả theo tham số (phải hashable: `int`, `tuple`, ...). Dùng để biến đệ quy luỹ thừa thành đệ quy theo số trạng thái.

## 4. Ý tưởng nền tảng / trực giác

### 4.1. Luật chơi diễn đạt bằng XOR

Gọi `X` là XOR của bảng ở đầu lượt.
- Nếu `X = 0` → người đi **thắng ngay**.
- Ngược lại, người đi chọn `aᵢ` để xoá; XOR mới là `X ^ aᵢ`. Nước đi **tự sát** nếu `X ^ aᵢ = 0`, tức `aᵢ = X`. Mọi nước khác là "an toàn" — chuyển lượt cho đối thủ với `X' = X ^ aᵢ ≠ 0`.

### 4.2. Tính tay ví dụ 1: `[1, 1, 2]`, `X = 2`

Alice:
- Xoá `2` (= X) → tự sát → thua.
- Xoá `1` → `[1, 2]`, `X' = 3`. Bob: xoá `1` → `[2]` (X = 2 ≠ 0, an toàn) hoặc xoá `2` → `[1]` (an toàn). Bob xoá một cái; Alice đối mặt `[2]` hoặc `[1]`, chỉ có một số, xoá nó → bảng rỗng XOR 0 → Alice thua.

→ Alice thua dù đi thế nào. `False` ✔.

### 4.3. Quan sát nhỏ

- Một phần tử `[v]` với `v ≠ 0`: người đi buộc xoá → bảng rỗng → thua.
- Hai phần tử `[a, b]` với `a ^ b ≠ 0`: `a ≠ b`, nên xoá `a` cho XOR mới `b`; nếu `b ≠ 0` thì an toàn và đối thủ rơi vào thế một phần tử → đối thủ thua. Nếu `b = 0` thì xoá `b` (XOR mới `a ≠ 0`). Luôn thắng!

Có vẻ **tính chẵn lẻ của `n`** đóng vai trò quan trọng. Ta sẽ để máy kiểm tra.

## 5. Cách 1 — Brute Force: minimax đệ quy thuần

**Ý tưởng.** `win(arr)`: nếu XOR = 0 → `True`. Ngược lại, thử xoá từng phần tử; nước đi tự sát bỏ qua; nếu có nước dẫn tới `not win(rest)` → `True`. Không có → `False`.

Không memo → độ phức tạp `O(n!)`, chỉ cho `n ≤ 8`.

In [5]:
def win_plain(arr):
    if reduce(xor, arr, 0) == 0:
        return True
    for i in range(len(arr)):
        rest = arr[:i] + arr[i + 1:]
        if reduce(xor, rest, 0) == 0:
            continue                       # nước tự sát
        if not win_plain(rest):
            return True
    return False

def xor_game_plain(nums):
    return win_plain(list(nums))

for a, exp in EXAMPLES:
    print(a, xor_game_plain(a), "(kỳ vọng", exp, ")")
    assert xor_game_plain(a) == exp

[1, 1, 2] False (kỳ vọng False )
[0, 1] True (kỳ vọng True )
[1, 2, 3] True (kỳ vọng True )
[5] False (kỳ vọng False )
[0] True (kỳ vọng True )
[2, 2, 2] False (kỳ vọng False )


Lưu ý `continue` cho nước tự sát: nếu **mọi** nước đều tự sát (hoặc dẫn tới thế thắng của đối thủ), vòng lặp kết thúc và trả `False` — người chơi buộc phải đi một nước thua. Điều này khớp luật "phải xoá một số".

**Độ phức tạp.** `O(n! · n)`. **Nút thắt:** cùng một tập phần tử còn lại được tính lại rất nhiều lần (xoá `a` rồi `b` hay `b` rồi `a` cho cùng trạng thái) → memo.

## 6. Bước đột phá

**Bước 1 — Memo trên bitmask** giảm xuống `O(2ⁿ · n)`: đủ để dò quy luật với `n ≤ 12`.

**Bước 2 — Dò quy luật bằng máy:** chạy trên hàng nghìn mảng nhỏ và so với giả thuyết `X == 0 or n % 2 == 0`.

**Bước 3 — Chứng minh (insight chính).**
- *Nếu* `X = 0` ở đầu lượt *thì* người đi thắng (theo luật).
- *Nếu* `n` chẵn và `X ≠ 0` *thì* người đi **luôn có nước an toàn**. Phản chứng: giả sử mọi nước đều tự sát, tức `aᵢ = X` với **mọi** `i`. Khi đó `X = a₁ ^ ... ^ aₙ = X ^ X ^ ... ^ X` (`n` lần, `n` chẵn) `= 0` — mâu thuẫn.
- Người có `n` chẵn đi một nước an toàn → đối thủ nhận `n − 1` (lẻ) và `X' ≠ 0`. Đối thủ đi bất kỳ nước an toàn nào (nếu có) → ta lại nhận `n` chẵn, `X ≠ 0` (nếu `X = 0` thì ta thắng luôn). Ta **không bao giờ** bị buộc tự sát; trò chơi hữu hạn → đối thủ thua.

**Kết luận:** Alice thắng ⇔ `X = 0` **hoặc** `n` chẵn.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Minimax + memo trên bitmask

In [6]:
def xor_game_mask(nums):
    n = len(nums)
    full = (1 << n) - 1

    @lru_cache(maxsize=None)
    def win(mask, x):
        # x = XOR của các phần tử trong mask (truyền theo để khỏi tính lại)
        if x == 0:
            return True
        m = mask
        while m:
            low = m & -m
            i = low.bit_length() - 1
            m ^= low
            nx = x ^ nums[i]
            if nx == 0:
                continue
            if not win(mask ^ low, nx):
                return True
        return False

    return win(full, reduce(xor, nums, 0))

for a, exp in EXAMPLES:
    assert xor_game_mask(a) == exp
print("Memo bitmask: OK")

Memo bitmask: OK


Kỹ thuật `low = m & -m` lấy **bit thấp nhất** đang bật, `low.bit_length() − 1` là chỉ số của nó. Duyệt các bit bật theo cách này nhanh hơn `for i in range(n)`.

Vì `x` hoàn toàn xác định bởi `mask`, đưa `x` vào khoá memo không làm tăng số trạng thái.

Code dạng LeetCode (chỉ chạy được với `n` nhỏ — để minh hoạ):

In [7]:
class SolutionMask:
    def xorGame(self, nums: List[int]) -> bool:
        n = len(nums)
        memo = {}

        def win(mask, x):
            if x == 0:
                return True
            if mask in memo:
                return memo[mask]
            res = False
            for i in range(n):
                if mask >> i & 1 and x ^ nums[i] != 0 and not win(mask ^ (1 << i), x ^ nums[i]):
                    res = True
                    break
            memo[mask] = res
            return res

        return win((1 << n) - 1, reduce(xor, nums, 0))

for a, exp in EXAMPLES:
    assert SolutionMask().xorGame(a) == exp
print("SolutionMask: OK")

SolutionMask: OK


**Độ phức tạp cách 2:** `O(2ⁿ · n)` thời gian, `O(2ⁿ)` bộ nhớ.

### 7.2. Cách 3 — Minimax + memo trên multiset chuẩn hoá

Các phần tử **bằng nhau** có thể hoán đổi → trạng thái chỉ cần là **multiset** (tuple đã sắp xếp). Khi có nhiều giá trị trùng, số trạng thái giảm mạnh.

In [8]:
def xor_game_multiset(nums):
    @lru_cache(maxsize=None)
    def win(state):                        # state: tuple đã sắp xếp
        x = reduce(xor, state, 0)
        if x == 0:
            return True
        tried = set()
        for i, v in enumerate(state):
            if v in tried:                 # xoá các bản sao giống nhau cho cùng kết quả
                continue
            tried.add(v)
            if x ^ v == 0:
                continue
            if not win(state[:i] + state[i + 1:]):
                return True
        return False

    return win(tuple(sorted(nums)))

for a, exp in EXAMPLES:
    assert xor_game_multiset(a) == exp
print("Memo multiset: OK")

Memo multiset: OK


**Độ phức tạp cách 3:** số multiset con ≤ `Π (cᵥ + 1)` với `cᵥ` là số lần xuất hiện của giá trị `v` — tốt hơn `2ⁿ` khi nhiều giá trị trùng, nhưng vẫn luỹ thừa trong trường hợp xấu.

### 7.3. Dò quy luật bằng máy

In [9]:
import random

def hypothesis(nums):
    return reduce(xor, nums, 0) == 0 or len(nums) % 2 == 0

rnd = random.Random(810)
mismatch = 0
stats = {}
for _ in range(3000):
    n = rnd.randint(1, 9)
    a = [rnd.randint(0, rnd.choice([1, 3, 7, 15])) for _ in range(n)]
    res = xor_game_mask(a)
    key = (n % 2, reduce(xor, a, 0) == 0)
    stats.setdefault(key, set()).add(res)
    mismatch += res != hypothesis(a)
print("số lần lệch giả thuyết:", mismatch)
for (par, x0), vals in sorted(stats.items()):
    print(f"n {'lẻ' if par else 'chẵn'}, XOR==0: {x0!s:5} -> các kết quả gặp: {vals}")

số lần lệch giả thuyết: 0
n chẵn, XOR==0: False -> các kết quả gặp: {True}
n chẵn, XOR==0: True  -> các kết quả gặp: {True}
n lẻ, XOR==0: False -> các kết quả gặp: {False}
n lẻ, XOR==0: True  -> các kết quả gặp: {True}


Bảng cho thấy rõ: khi `n` chẵn **hoặc** `XOR = 0` thì luôn `True`; khi `n` lẻ và `XOR ≠ 0` thì luôn `False`.

### 7.4. Cách 4 — Công thức `O(n)`

In [10]:
def xor_game_math(nums):
    return reduce(xor, nums, 0) == 0 or len(nums) % 2 == 0

class SolutionMath:
    def xorGame(self, nums: List[int]) -> bool:
        x = 0
        for v in nums:
            x ^= v
        return x == 0 or len(nums) % 2 == 0

for a, exp in EXAMPLES:
    assert xor_game_math(a) == exp and SolutionMath().xorGame(a) == exp
print("Công thức: OK")

Công thức: OK


**Độ phức tạp cách 4:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Trace cây trò chơi (minimax) cho `[1, 1, 2]`

In [11]:
def trace_game(arr, depth=0, player="Alice"):
    pad = "  " * depth
    x = reduce(xor, arr, 0)
    other = "Bob" if player == "Alice" else "Alice"
    if x == 0:
        print(f"{pad}{player} gặp {arr}, XOR=0 -> {player} THẮNG")
        return True
    print(f"{pad}{player} gặp {arr}, XOR={x}")
    can_win = False
    for i in range(len(arr)):
        rest = arr[:i] + arr[i + 1:]
        if reduce(xor, rest, 0) == 0:
            print(f"{pad}  xoá {arr[i]} -> {rest} XOR=0 : tự sát (thua)")
            continue
        print(f"{pad}  xoá {arr[i]} -> {rest}")
        if not trace_game(rest, depth + 2, other):
            can_win = True
            print(f"{pad}  => {player} thắng bằng cách xoá {arr[i]}")
            break
    if not can_win:
        print(f"{pad}  => {player} THUA ở {arr}")
    return can_win

trace_game([1, 1, 2])

Alice gặp [1, 1, 2], XOR=2
  xoá 1 -> [1, 2]
    Bob gặp [1, 2], XOR=3
      xoá 1 -> [2]
        Alice gặp [2], XOR=2
          xoá 2 -> [] XOR=0 : tự sát (thua)
          => Alice THUA ở [2]
      => Bob thắng bằng cách xoá 1
  xoá 1 -> [1, 2]
    Bob gặp [1, 2], XOR=3
      xoá 1 -> [2]
        Alice gặp [2], XOR=2
          xoá 2 -> [] XOR=0 : tự sát (thua)
          => Alice THUA ở [2]
      => Bob thắng bằng cách xoá 1
  xoá 2 -> [1, 1] XOR=0 : tự sát (thua)
  => Alice THUA ở [1, 1, 2]


False

### 8.2. Trace chiến lược "luôn đi nước an toàn" khi `n` chẵn

Mô phỏng Alice dùng chiến lược của chứng minh (chọn nước an toàn **bất kỳ**), Bob chơi **tối ưu** (dùng minimax). Ta in bảng trạng thái từng lượt.

In [12]:
def play_out(nums, seed=0):
    r = random.Random(seed)
    board = list(nums)
    turn = "Alice"
    print(f"{'lượt':>4} | {'người':5} | {'bảng':22} | XOR | nước đi")
    t = 0
    while True:
        x = reduce(xor, board, 0)
        if x == 0:
            print(f"{t:>4} | {turn:5} | {str(board):22} | {x:3} | XOR=0 đầu lượt -> {turn} thắng")
            return turn
        safe = [i for i, v in enumerate(board) if x ^ v != 0]
        if not safe:
            i = 0
            print(f"{t:>4} | {turn:5} | {str(board):22} | {x:3} | không có nước an toàn, buộc xoá {board[i]} -> {turn} thua")
            return "Bob" if turn == "Alice" else "Alice"
        if turn == "Alice":
            i = r.choice(safe)                                   # bất kỳ nước an toàn
        else:
            good = [j for j in safe if not xor_game_mask(board[:j] + board[j + 1:])]
            i = good[0] if good else safe[0]                     # Bob tối ưu
        before = board[:]
        v = board.pop(i)
        print(f"{t:>4} | {turn:5} | {str(before):22} | {x:3} | xoá {v}")
        turn = "Bob" if turn == "Alice" else "Alice"
        t += 1

winner = play_out([3, 5, 6, 1, 2, 7])
print("Người thắng:", winner)

lượt | người | bảng                   | XOR | nước đi
   0 | Alice | [3, 5, 6, 1, 2, 7]     |   4 | xoá 1
   1 | Bob   | [3, 5, 6, 2, 7]        |   5 | xoá 3
   2 | Alice | [5, 6, 2, 7]           |   6 | xoá 2
   3 | Bob   | [5, 6, 7]              |   4 | xoá 5
   4 | Alice | [6, 7]                 |   1 | xoá 6
   5 | Bob   | [7]                    |   7 | không có nước an toàn, buộc xoá 7 -> Bob thua
Người thắng: Alice


Bảng trạng thái tóm tắt quy luật: ở các lượt của Alice, số phần tử luôn **chẵn** (6, 4, 2) và XOR ≠ 0 → luôn có nước an toàn. Ở lượt của Bob, số phần tử lẻ; cuối cùng Bob phải đối mặt một phần tử khác 0 (hoặc buộc tự sát).

Thêm một ván nữa, lần này **cả hai** chọn ngẫu nhiên một nước an toàn — Alice vẫn thắng:

In [13]:
def play_table(nums, seed=1):
    r = random.Random(seed)
    board, turn, rows = list(nums), "Alice", []
    while True:
        x = reduce(xor, board, 0)
        if x == 0:
            rows.append((turn, board[:], x, "XOR=0 -> thắng"))
            break
        safe = [i for i, v in enumerate(board) if x ^ v != 0]
        if not safe:
            rows.append((turn, board[:], x, "buộc tự sát -> thua"))
            break
        i = r.choice(safe)
        rows.append((turn, board[:], x, f"xoá {board[i]}"))
        board.pop(i)
        turn = "Bob" if turn == "Alice" else "Alice"
    for k, (p, b, x, mv) in enumerate(rows):
        print(f"{k:>2} | {p:5} | n={len(b)} | {str(b):20} | XOR={x:2} | {mv}")

play_table([3, 5, 6, 1, 2, 7])

 0 | Alice | n=6 | [3, 5, 6, 1, 2, 7]   | XOR= 4 | xoá 5
 1 | Bob   | n=5 | [3, 6, 1, 2, 7]      | XOR= 1 | xoá 3
 2 | Alice | n=4 | [6, 1, 2, 7]         | XOR= 2 | xoá 1
 3 | Bob   | n=3 | [6, 2, 7]            | XOR= 3 | xoá 6
 4 | Alice | n=2 | [2, 7]               | XOR= 5 | xoá 7
 5 | Bob   | n=1 | [2]                  | XOR= 2 | buộc tự sát -> thua


(Ở đây cả hai đều chọn ngẫu nhiên nước an toàn; dù Bob chơi thế nào, Alice — người giữ `n` chẵn — không bao giờ bị buộc tự sát.)

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def xorGame(self, nums: List[int]) -> bool:
        x = 0
        for v in nums:
            x ^= v
        # XOR = 0: Alice thắng ngay; n chẵn: Alice luôn có nước an toàn
        return x == 0 or len(nums) % 2 == 0

for a, exp in EXAMPLES:
    assert Solution().xorGame(a) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Định lý.** Ở đầu lượt với bảng gồm `n` số, XOR `X`:
- (a) `X = 0` → người đi thắng.
- (b) `X ≠ 0`, `n` chẵn → người đi thắng.
- (c) `X ≠ 0`, `n` lẻ → người đi thua.

**Chứng minh (quy nạp theo `n`).**

(a) Theo luật.

(b) Có ít nhất một nước an toàn (phản chứng ở mục 6: nếu mọi `aᵢ = X` thì `X = X·[n lẻ] = 0`). Đi nước an toàn → đối thủ có `n − 1` (lẻ) phần tử và `X' ≠ 0` → theo (c) với `n − 1` thì đối thủ thua.

(c) `n` lẻ, `X ≠ 0`. Mọi nước: hoặc tự sát (thua ngay), hoặc an toàn → đối thủ có `n − 1` (chẵn) và `X' ≠ 0` → theo (b) đối thủ thắng. Vậy người đi thua. Trường hợp cơ sở `n = 1`, `X = a₁ ≠ 0`: chỉ có nước xoá `a₁` → bảng rỗng XOR 0 → tự sát → thua. ✔

**Kết luận cho Alice:** thắng ⇔ (a) hoặc (b) ⇔ `X = 0 or n % 2 == 0`. ∎

(Chú ý (b) dùng (c) với `n − 1`, và (c) dùng (b) với `n − 1` — quy nạp đồng thời, cơ sở `n = 1` cho (c) và `n = 0` không xảy ra cho (b) vì `X = 0` khi bảng rỗng.)

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Minimax thuần | `O(n! · n)` | `O(n)` ngăn xếp |
| 2. Memo bitmask | `O(2ⁿ · n)` | `O(2ⁿ)` |
| 3. Memo multiset | `≤ O(Π(cᵥ+1) · n)` | tương ứng |
| 4. Công thức | `O(n)` | `O(1)` |

In [15]:
import time
for n in (8, 10, 12, 14):
    a = [rnd.randint(1, 2**16 - 1) for _ in range(n)]
    t0 = time.perf_counter()
    r1 = xor_game_mask(a)
    el = time.perf_counter() - t0
    print(f"n={n:2}: memo bitmask {el*1000:7.1f} ms -> {r1}, công thức -> {xor_game_math(a)}")

n= 8: memo bitmask     0.0 ms -> True, công thức -> True
n=10: memo bitmask     0.1 ms -> True, công thức -> True
n=12: memo bitmask     0.3 ms -> True, công thức -> True
n=14: memo bitmask     1.3 ms -> True, công thức -> True


Với `n = 1000` (giới hạn đề), chỉ công thức khả thi.

## 12. Bộ test

In [16]:
methods = {
    "plain": xor_game_plain,
    "mask": xor_game_mask,
    "multiset": xor_game_multiset,
    "math": xor_game_math,
    "SolutionMask": lambda a: SolutionMask().xorGame(a),
    "SolutionMath": lambda a: SolutionMath().xorGame(a),
    "Solution": lambda a: Solution().xorGame(a),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([0, 0, 0], True),          # XOR = 0
    ([7, 7], True),             # XOR = 0, n chẵn
    ([1, 2], True),
    ([1, 2, 4], False),         # n lẻ, XOR = 7
    ([3, 3, 3], False),
    ([0, 0, 5], False),
]
for name, fn in methods.items():
    for a, exp in edge:
        assert fn(a) == exp, (name, a)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed, n nhỏ: đối chiếu minimax thuần (n <= 7) và memo
for seed in range(1500):
    r = random.Random(seed)
    n = r.randint(1, 7)
    a = [r.randint(0, r.choice([1, 3, 15, 2**16 - 1])) for _ in range(n)]
    ref = xor_game_plain(a)
    for name, fn in methods.items():
        assert fn(a) == ref, (name, a)
print("1500 mảng nhỏ: khớp minimax thuần")

# 12.3 n vừa (8..12): memo bitmask/multiset khớp công thức
for seed in range(200):
    r = random.Random(4000 + seed)
    n = r.randint(8, 12)
    a = [r.randint(0, r.choice([3, 7, 255])) for _ in range(n)]
    ref = xor_game_mask(a)
    assert xor_game_multiset(a) == ref == Solution().xorGame(a), a
print("n = 8..12: OK")

# 12.4 n lớn: chỉ công thức, kiểm tra tính chất
for seed in range(200):
    r = random.Random(9000 + seed)
    n = r.randint(1, 1000)
    a = [r.randrange(2**16) for _ in range(n)]
    assert Solution().xorGame(a) == (reduce(xor, a, 0) == 0 or n % 2 == 0)
print("n lớn: OK")

Ví dụ + edge cases: OK


1500 mảng nhỏ: khớp minimax thuần


n = 8..12: OK
n lớn: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét tính chẵn lẻ, quên trường hợp XOR = 0

**Code sai:**

In [17]:
def wrong_parity_only(nums):
    return len(nums) % 2 == 0          # SAI: quên X == 0

for a in ([1, 2, 3], [0], [5, 5, 0]):
    print(a, "sai:", wrong_parity_only(a), " đúng:", Solution().xorGame(a))

[1, 2, 3] sai: False  đúng: True
[0] sai: False  đúng: True
[5, 5, 0] sai: False  đúng: True


**Vì sao sai:** nếu XOR ban đầu đã bằng 0, Alice thắng ngay ở lượt đầu dù `n` lẻ.

**Code đúng:** `x == 0 or n % 2 == 0`.

### Lỗi 2 — Minimax hiểu sai luật: coi nước tự sát là "đối thủ thắng ở lượt sau"

**Code sai:**

In [18]:
def wrong_rule(arr):
    if not arr:
        return True                     # SAI: mô hình bảng rỗng như một lượt bình thường
    if reduce(xor, arr, 0) == 0:
        return True
    return any(not wrong_rule(arr[:i] + arr[i + 1:]) for i in range(len(arr)))

for a in ([1, 1, 2], [5], [2, 2, 2]):
    print(a, "sai:", wrong_rule(a), " đúng:", Solution().xorGame(a))

[1, 1, 2] sai: False  đúng: False
[5] sai: False  đúng: False
[2, 2, 2] sai: False  đúng: False


**Vì sao sai (và vì sao ở đây vẫn đúng):** nước "tự sát" làm XOR = 0, nên đối thủ bắt đầu lượt với XOR = 0 → đối thủ thắng; `wrong_rule` mô hình hoá điều này qua `not wrong_rule(rest)` = `not True` = `False` — thực ra **tương đương** với luật. Bài học: hai cách diễn đạt luật ("người xoá thua" và "người kế tiếp gặp XOR 0 thì thắng") là **cùng một luật**; nhưng phải cẩn thận trường hợp bảng rỗng (`XOR([]) = 0`). Nếu ai đó viết `if not arr: return False` ("không còn gì để xoá thì thua") thì sai:

In [19]:
def wrong_empty(arr):
    if not arr:
        return False                    # SAI: bảng rỗng có XOR 0 -> người đi thắng
    x = reduce(xor, arr, 0)
    if x == 0 and arr:
        return True
    return any(not wrong_empty(arr[:i] + arr[i + 1:]) for i in range(len(arr)))

for a in ([5], [1, 1, 2], [3, 5, 6]):
    print(a, "sai:", wrong_empty(a), " đúng:", Solution().xorGame(a))

[5] sai: True  đúng: False
[1, 1, 2] sai: True  đúng: False
[3, 5, 6] sai: True  đúng: True


**Code đúng:** coi bảng rỗng có XOR = 0 (dùng `reduce(xor, arr, 0)` với giá trị khởi tạo 0), hoặc bỏ qua nước tự sát như cách 1.

### Lỗi 3 — Nghĩ rằng "Alice nên xoá số lớn nhất / nhỏ nhất" (tham lam)

**Code sai:**

In [20]:
def wrong_greedy(nums):
    board, turn = sorted(nums), 0       # 0 = Alice
    while True:
        x = reduce(xor, board, 0)
        if x == 0:
            return turn == 0
        safe = [v for v in board if x ^ v != 0]
        if not safe:
            return turn == 1
        v = max(safe)                   # SAI: cả hai đều tham lam xoá số lớn nhất an toàn
        board.remove(v)
        turn ^= 1

cnt = sum(wrong_greedy(a) != Solution().xorGame(a) for a in
          ([random.Random(s).randint(0, 7) for _ in range(random.Random(s).randint(1, 7))] for s in range(2000)))
print("số input mà mô phỏng tham lam cho kết quả khác:", cnt)

số input mà mô phỏng tham lam cho kết quả khác: 0


**Vì sao sai (hoặc không cần):** mô phỏng với một chiến lược cố định cho **cả hai** người không phản ánh "chơi tối ưu" — Bob có thể chọn nước khác. Ở bài này, kết quả tình cờ không phụ thuộc chiến lược (do định lý), nên mô phỏng tham lam có thể ra đúng, nhưng đó là **may mắn**, không phải lập luận. Lời giải đúng cần chứng minh như mục 10.

**Code đúng:** công thức đã chứng minh.

## 14. Mẹo & nhận dạng pattern

- **Trò chơi hai người, n lớn (1000), giá trị nhỏ** → gần như chắc chắn có **công thức**; dùng minimax + memo trên input nhỏ để **dò quy luật**, rồi chứng minh.
- **XOR "bỏ một phần tử"** = `X ^ aᵢ`.
- **Phản chứng kiểu "nếu mọi nước đều xấu thì…"** là công cụ mạnh để chứng minh tồn tại nước tốt.
- **Chiến lược bảo toàn bất biến** (ở đây: "đến lượt tôi thì n chẵn và X ≠ 0") → người giữ bất biến thắng.
- Cùng họ: **292 Nim Game**, **877 Stone Game** (Alice luôn thắng với số đống chẵn), **1025 Divisor Game** (thắng ⇔ n chẵn), **Nim cổ điển** (thắng ⇔ XOR ≠ 0).

## 15. Biến thể

1. **Luật ngược: người làm XOR = 0 thắng.** Phân tích lại bằng minimax nhỏ để dò.
2. **Thay XOR bằng tổng mod `m`:** điều kiện tự sát `aᵢ ≡ S (mod m)`; lập luận phản chứng: mọi `aᵢ ≡ S` ⇒ `S ≡ n·S` ⇒ `(n−1)S ≡ 0`.
3. **Mỗi lượt xoá 1 hoặc 2 số:** trạng thái phức tạp hơn; dò bằng memo.
4. **Nim chuẩn** (bốc tuỳ ý từ một đống): người đi thắng ⇔ XOR các đống ≠ 0.
5. **Hỏi số nước đi an toàn ở lượt đầu:** `#{i : aᵢ ≠ X}`.

Minh hoạ biến thể 1 — dò quy luật bằng máy:

In [21]:
def win_reverse(nums):
    """Biến thể: ai xoá làm XOR còn lại = 0 thì THẮNG; bảng rỗng ở đầu lượt -> người đi thua (không đi được)."""
    @lru_cache(None)
    def win(state):
        if not state:
            return False
        x = reduce(xor, state, 0)
        for i, v in enumerate(state):
            if x ^ v == 0:
                return True                       # xoá v làm XOR = 0 -> thắng ngay
        return any(not win(state[:i] + state[i + 1:]) for i in range(len(state)))
    return win(tuple(sorted(nums)))

pattern = {}
for s in range(1500):
    r = random.Random(s)
    a = [r.randint(0, 7) for _ in range(r.randint(1, 7))]
    key = (len(a) % 2, reduce(xor, a, 0) == 0, any(v == reduce(xor, a, 0) for v in a))
    pattern.setdefault(key, set()).add(win_reverse(a))
for k, v in sorted(pattern.items()):
    print(f"n lẻ={k[0]}, XOR==0={k[1]!s:5}, có aᵢ==X={k[2]!s:5} -> {v}")

n lẻ=0, XOR==0=False, có aᵢ==X=False -> {False}
n lẻ=0, XOR==0=False, có aᵢ==X=True  -> {True}
n lẻ=0, XOR==0=True , có aᵢ==X=False -> {False}
n lẻ=0, XOR==0=True , có aᵢ==X=True  -> {True}
n lẻ=1, XOR==0=False, có aᵢ==X=False -> {True}
n lẻ=1, XOR==0=False, có aᵢ==X=True  -> {True}
n lẻ=1, XOR==0=True , có aᵢ==X=False -> {True}
n lẻ=1, XOR==0=True , có aᵢ==X=True  -> {True}


Đọc bảng (trên dữ liệu thử): với luật ngược, **n lẻ → người đi luôn thắng**; **n chẵn → thắng ⇔ có phần tử bằng X** (xoá ngay để thắng). Đây chỉ là quy luật *quan sát được* — muốn dùng phải chứng minh tương tự mục 10 (gợi ý: khi n lẻ, nếu không thắng ngay thì luôn có nước khiến đối thủ, với n chẵn, không có phần tử nào bằng XOR mới).

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax thuần | thử mọi nước | `O(n!·n)` | `O(n)` | hiểu luật, `n ≤ 7` |
| 2. Memo bitmask | trạng thái = tập còn lại | `O(2ⁿ·n)` | `O(2ⁿ)` | dò quy luật, `n ≤ 15` |
| 3. Memo multiset | gộp phần tử trùng | `≤ O(Π(cᵥ+1)·n)` | tương ứng | nhiều giá trị trùng |
| 4. Công thức | `X == 0 or n chẵn` | `O(n)` | `O(1)` | **nộp bài** |

## 17. Bài tập

**Bài 1 (đoán output).** `xorGame([1, 2, 4, 8])`, `xorGame([1, 2, 4])`, `xorGame([6, 6, 6, 6, 6])` = ?

**Bài 2 (tính tay).** Với `[3, 3, 5]`: liệt kê nước đi của Alice, nước nào tự sát? Ai thắng?

**Bài 3 (điền TODO).** Hoàn thiện `safe_moves(nums)` trả về danh sách chỉ số mà Alice có thể xoá **không tự sát** ở lượt đầu.

**Bài 4 (tự viết lại).** Viết `optimal_move(nums)` trả về một chỉ số Alice nên xoá để **thắng** (nếu Alice thắng và `X ≠ 0`), hoặc `None`. Kiểm tra bằng minimax rằng sau nước đó Bob ở thế thua.

**Bài 5 (mini challenge).** Biến thể 2: thay XOR bằng **tổng modulo `m`** (người làm tổng còn lại ≡ 0 mod m thì thua; đầu lượt tổng ≡ 0 thì thắng). Dò quy luật bằng memo và viết công thức (gợi ý: phân tích `(n − 1)·S ≡ 0 (mod m)`), kiểm tra với `m = 2, 3, 4`.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def safe_moves(nums):
    # TODO: X = XOR tất cả; trả các i với nums[i] != X
    pass

# Bài 4 — TODO
def optimal_move(nums):
    # TODO: nếu X == 0 hoặc n lẻ -> None (thắng sẵn / không thắng); ngược lại trả bất kỳ i an toàn
    pass

# Bài 5 — TODO
def sum_mod_game(nums, m):
    # TODO: minimax memo với điều kiện tổng % m
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tính XOR: `1^2^4^8 = 15`, `1^2^4 = 7`, `6^6^6^6^6 = 6`.
- Mức 2: `n` lần lượt 4, 3, 5.
- Mức 3: `True`, `False`, `False`.

**Bài 2.**
- Mức 1: `X = 3 ^ 3 ^ 5 = 5`.
- Mức 2: xoá `5` (= X) là tự sát; xoá `3` → `[3, 5]`, XOR 6.
- Mức 3: Bob với `[3, 5]` (n chẵn) thắng → Alice thua (`n = 3` lẻ, `X ≠ 0`).

**Bài 3.**
- Mức 1: nước tự sát ⇔ `nums[i] == X`.
- Mức 2: dùng list comprehension.
- Mức 3: với `n` chẵn, `X ≠ 0`, danh sách khác rỗng.

**Bài 4.**
- Mức 1: theo chứng minh, **mọi** nước an toàn đều thắng khi `n` chẵn.
- Mức 2: trả `safe_moves(nums)[0]`.
- Mức 3: kiểm tra `not xor_game_mask(nums sau khi xoá)`.

**Bài 5.**
- Mức 1: viết lại minimax với `sum % m`.
- Mức 2: nếu mọi nước đều tự sát: `aᵢ ≡ S` ⇒ `S ≡ n·S` ⇒ `(n − 1)·S ≡ 0 (mod m)`.
- Mức 3: quy luật không còn chỉ phụ thuộc chẵn lẻ; để máy dò và so sánh.

## 19. Đáp án tham khảo

In [23]:
S = Solution()
# Bài 1
assert S.xorGame([1, 2, 4, 8]) is True
assert S.xorGame([1, 2, 4]) is False
assert S.xorGame([6, 6, 6, 6, 6]) is False

# Bài 2
assert S.xorGame([3, 3, 5]) is False and xor_game_plain([3, 3, 5]) is False

# Bài 3
def safe_moves_ans(nums):
    X = reduce(xor, nums, 0)
    return [i for i, v in enumerate(nums) if v != X]

assert safe_moves_ans([3, 3, 5]) == [0, 1]
for s in range(500):
    r = random.Random(s)
    a = [r.randint(0, 15) for _ in range(r.choice([2, 4, 6, 8]))]
    if reduce(xor, a, 0) != 0:
        assert safe_moves_ans(a), a          # n chẵn, X != 0 -> luôn có nước an toàn

# Bài 4
def optimal_move_ans(nums):
    X = reduce(xor, nums, 0)
    if X == 0 or len(nums) % 2 == 1:
        return None
    return safe_moves_ans(nums)[0]

for s in range(500):
    r = random.Random(s)
    a = [r.randint(0, 15) for _ in range(r.randint(1, 9))]
    mv = optimal_move_ans(a)
    if mv is not None:
        rest = a[:mv] + a[mv + 1:]
        assert reduce(xor, rest, 0) != 0 and not xor_game_mask(rest)

# Bài 5
def sum_mod_game_ans(nums, m):
    @lru_cache(None)
    def win(state):
        s = sum(state) % m
        if s == 0:
            return True
        for i in range(len(state)):
            rest = state[:i] + state[i + 1:]
            if sum(rest) % m == 0:
                continue
            if not win(rest):
                return True
        return False
    return win(tuple(sorted(nums)))

for m in (2, 3, 4):
    table = {}
    for s in range(800):
        r = random.Random(s)
        a = [r.randint(0, 9) for _ in range(r.randint(1, 7))]
        key = (len(a) % 2, sum(a) % m == 0)
        table.setdefault(key, set()).add(sum_mod_game_ans(a, m))
    print(f"m={m}:", {k: v for k, v in sorted(table.items())})
# Với m = 2: tổng mod 2 chính là XOR của các bit thấp -> cùng quy luật chẵn lẻ
for s in range(500):
    r = random.Random(s)
    a = [r.randint(0, 1) for _ in range(r.randint(1, 8))]
    assert sum_mod_game_ans(a, 2) == xor_game_math(a)
print("Tất cả đáp án đúng")

m=2: {(0, False): {True}, (0, True): {True}, (1, False): {False}, (1, True): {True}}


m=3: {(0, False): {False, True}, (0, True): {True}, (1, False): {False, True}, (1, True): {True}}
m=4: {(0, False): {True}, (0, True): {True}, (1, False): {False}, (1, True): {True}}
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
x = 0
for v in nums: x ^= v
return x == 0 or len(nums) % 2 == 0
```

**Template dò quy luật trò chơi:**

```python
@lru_cache(None)
def win(state):
    if điều_kiện_thắng_ngay(state): return True
    return any(not win(next_state) for next_state in các_nước_hợp_lệ(state))
```

**Checklist trước khi submit:**

- [ ] Xét **cả** `XOR == 0` lẫn `n` chẵn.
- [ ] XOR bắt đầu từ `0` (bảng rỗng có XOR 0).
- [ ] Không mô phỏng trò chơi với `n = 1000` (luỹ thừa).
- [ ] Hiểu và tự chứng minh được vì sao `n` chẵn luôn có nước an toàn.